%md
# Tabela de Monitoramento de atividades usando IA: ai_usage
A tabela de monitoramento de atividades usando IA considera qualquer tipo de ação realizada no ambiente *Databricks* que conte com a utilização de recursos como **Genie** ou **Assistant**.


Por isso é importante entender que ambos os recursos de IA da plataforma, possuem caracteristicas distintas e cada um é armazenado em uma fonte de dados diferente. As ocorrencias do serviço "Genie" são armazenados na tabela sistemica **system.access.audit**, enquanto os registros de uso dos recursos do "Assistant" ficam localizados na tabela sistemia **system.access.assistant_events**.

## Pipeline:ai_usage
**Pipeline** de ingestão de registros de utilização de IA no ambiente Databricks, considerando os recursos **Genie** e **Assistant**, seguindo os seguintes passos:
- Exclui a tabela, se ela existir;
- Realiza a criação da tabela;
- Atribui ao campo "email" a caracteristica de chave estrangeira;
- Realiza o streaming de dados com a origem da informação;
- Realiza a insersão dos valores na tabela gerada.

In [0]:
%sql
DROP TABLE IF EXISTS data_governance.ai_usage;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS  workspace.data_governance.ai_usage(
  email STRING NOT NULL
  , service_name STRING
  , event_date DATE
);

In [0]:
%sql
ALTER TABLE data_governance.ai_usage ADD CONSTRAINT fk_email_ai FOREIGN KEY (email) REFERENCES data_governance.users(email)

In [0]:
%sql
COMMENT ON TABLE data_governance.ai_usage is '## Content
AI resources usage information, containing information about the AI resources used by each user. This table up to this moment focus only into Genie and AI Assistant usage.
### Update
This table updates every day 7:00 AM (UTC-03:00)';
COMMENT ON COLUMN data_governance.ai_usage.email is 'User email';
COMMENT ON COLUMN data_governance.ai_usage.service_name is 'AI resource name, it can be Genie or Assistant.';
COMMENT ON COLUMN data_governance.ai_usage.event_date is 'Date when the AI resource was used.';

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE data_governance.ai_usage(
  email STRING
  , service_name STRING
  , event_date DATE
) AS SELECT
  u.email
  , named_struct('service_name', ua.usage_log.service_name)
  , ua.event_date
FROM STREAM data_governance.users_usage ua
LEFT JOIN STREAM data_governance.users u ON ua.email = u.email;

In [0]:
%sql
INSERT INTO data_governance.ai_usage(
  email,
  service_name,
  event_date
)
WITH genie AS (
  SELECT
    u.email,
    ua.usage_log.service_name AS service_name,
    ua.event_date
  FROM data_governance.users_usage ua
  LEFT JOIN data_governance.users u ON ua.email = u.email
  WHERE ua.usage_log.service_name IN ('aibigenie', 'genie')
),
assistant AS (
  SELECT
    a.initiated_by AS email,
    'assistant' AS service_name,
    a.event_date
  FROM system.access.assistant_events a
)
SELECT
  COALESCE(g.email, a.email) AS email,
  COALESCE(g.service_name, a.service_name) AS service_name,
  COALESCE(g.event_date, a.event_date) AS event_date
FROM genie g
FULL OUTER JOIN assistant a
  ON g.email = a.email;